# Import Required Libraries

Import necessary libraries such as pandas, numpy, matplotlib, seaborn, and scikit-learn for data manipulation, analysis, and visualization.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load Raw Data

Load the raw e-commerce data from a CSV file into a pandas DataFrame and perform initial inspection of the data structure.

In [3]:
# Load the data
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

# Merge datasets to create comprehensive order data
# Orders with customers
df = orders.merge(customers, on='customer_id', how='left')

# Add order items (price, freight)
df = df.merge(order_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']], on='order_id', how='left')

# Add payments (sum payment_value per order)
payment_agg = order_payments.groupby('order_id')['payment_value'].sum().reset_index()
df = df.merge(payment_agg, on='order_id', how='left')

# Add reviews
df = df.merge(order_reviews[['order_id', 'review_score']], on='order_id', how='left')

# Add product categories
df = df.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
df = df.merge(category_translation, on='product_category_name', how='left')

# Rename columns for consistency
df = df.rename(columns={
    'order_purchase_timestamp': 'order_date',
    'order_delivered_customer_date': 'delivered_date',
    'order_estimated_delivery_date': 'estimated_delivery_date',
    'payment_value': 'total',
    'review_score': 'rating',
    'product_category_name_english': 'category'
})

# Select relevant columns and drop duplicates (since order_items can have multiple rows per order)
df = df[['order_id', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 
         'order_date', 'delivered_date', 'estimated_delivery_date', 'order_status', 
         'product_id', 'price', 'freight_value', 'total', 'rating', 'category']].drop_duplicates()

# Display first few rows
print("First 5 rows of the merged dataset:")
display(df.head())

# Basic info
print("\nDataset info:")
df.info()

print("\nDataset shape:", df.shape)

First 5 rows of the merged dataset:


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_date,delivered_date,estimated_delivery_date,order_status,product_id,price,freight_value,total,rating,category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,delivered,87285b34884572647811a353c7ac498a,29.99,8.72,38.71,4.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,delivered,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,141.46,4.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,delivered,aa4383b373c6aca5d8797843e5594415,159.90,19.22,179.12,5.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,delivered,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,72.20,5.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,delivered,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,28.62,5.0,stationery



Dataset info:
<class 'pandas.core.frame.DataFrame'>
Index: 103428 entries, 0 to 114091
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   order_id                 103428 non-null  object 
 1   customer_id              103428 non-null  object 
 2   customer_unique_id       103428 non-null  object 
 3   customer_city            103428 non-null  object 
 4   customer_state           103428 non-null  object 
 5   order_date               103428 non-null  object 
 6   delivered_date           100409 non-null  object 
 7   estimated_delivery_date  103428 non-null  object 
 8   order_status             103428 non-null  object 
 9   product_id               102652 non-null  object 
 10  price                    102652 non-null  float64
 11  freight_value            102652 non-null  float64
 12  total                    103427 non-null  float64
 13  rating                   102612 non-null  float64

# Data Cleaning and Preprocessing

Clean the data by handling missing values, duplicates, data type conversions, and feature engineering for analysis.

In [4]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Handle missing values
df = df.dropna(subset=['order_date', 'customer_id'])  # Drop rows with missing essential data
df['rating'] = df['rating'].fillna(df['rating'].median())  # Fill missing ratings with median
df['category'] = df['category'].fillna('unknown')  # Fill missing categories

# Convert date columns
df['order_date'] = pd.to_datetime(df['order_date'])

# Remove duplicates
df = df.drop_duplicates()

# Feature engineering
df['total_revenue'] = df['total']  # Already the total payment

# Filter for delivered orders only (successful purchases)
df = df[df['order_status'] == 'delivered']

print("Data after cleaning:")
print("Shape:", df.shape)
display(df.head())

# Save cleaned data
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print("Cleaned data saved to ../data/processed/cleaned_data.csv")

Missing values per column:
order_id                      0
customer_id                   0
customer_unique_id            0
customer_city                 0
customer_state                0
order_date                    0
delivered_date             3019
estimated_delivery_date       0
order_status                  0
product_id                  776
price                       776
freight_value               776
total                         1
rating                      816
category                   2262
dtype: int64
Data after cleaning:
Shape: (100410, 16)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_date,delivered_date,estimated_delivery_date,order_status,product_id,price,freight_value,total,rating,category,total_revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,delivered,87285b34884572647811a353c7ac498a,29.99,8.72,38.71,4.0,housewares,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,delivered,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,141.46,4.0,perfumery,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,delivered,aa4383b373c6aca5d8797843e5594415,159.90,19.22,179.12,5.0,auto,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,delivered,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,72.20,5.0,pet_shop,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,delivered,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,28.62,5.0,stationery,28.62


Cleaned data saved to ../data/processed/cleaned_data.csv


# Exploratory Data Analysis (EDA)

Perform EDA to understand data distributions, correlations, and key statistics using descriptive statistics and visualizations.

In [ ]:
# Descriptive statistics
print("Descriptive statistics:")
display(df.describe())

# Distribution of numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Funnel Analysis

Analyze the customer journey funnel, calculating drop-off rates at each stage and identifying bottlenecks.

In [ ]:
# Define funnel stages based on order status
# Since we filtered for delivered, let's analyze delivery time and ratings

# Calculate delivery time
df['delivery_time'] = (pd.to_datetime(df['order_delivered_customer_date']) - df['order_date']).dt.days

# Funnel analysis: orders -> delivered (successful)
total_orders = len(orders)
delivered_orders = len(df)
delivery_rate = delivered_orders / total_orders * 100

print(f"Total Orders: {total_orders}")
print(f"Delivered Orders: {delivered_orders}")
print(f"Delivery Rate: {delivery_rate:.1f}%")

# Visualize delivery time distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['delivery_time'].dropna(), bins=30, kde=True)
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time (days)')
plt.show()

# Average delivery time by state
avg_delivery_by_state = df.groupby('customer_state')['delivery_time'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
avg_delivery_by_state.plot(kind='bar')
plt.title('Average Delivery Time by State (Top 10)')
plt.ylabel('Average Delivery Time (days)')
plt.xticks(rotation=45)
plt.show()

# Customer Segmentation

Apply clustering techniques like K-means to segment customers based on behavior, demographics, or purchase patterns.

In [ ]:
# Prepare data for clustering
customer_features = df.groupby('customer_unique_id').agg({
    'total': 'sum',
    'order_id': 'count',
    'price': 'mean'
}).rename(columns={'order_id': 'num_orders', 'total': 'total_revenue', 'price': 'avg_price'})

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(customer_features)

# Apply K-means
kmeans = KMeans(n_clusters=4, random_state=42)
customer_features['cluster'] = kmeans.fit_predict(scaled_features)

print("Customer segments:")
display(customer_features.groupby('cluster').mean())

# Visualize clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(x='total_revenue', y='num_orders', hue='cluster', data=customer_features, palette='viridis')
plt.title('Customer Segmentation')
plt.show()

# Revenue and Conversion Analysis

Calculate key metrics such as conversion rates, average order value, customer lifetime value, and revenue trends over time.

In [ ]:
# Key metrics
total_revenue = df['total'].sum()
total_orders = df['order_id'].nunique()
avg_order_value = total_revenue / total_orders
total_customers = df['customer_unique_id'].nunique()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Orders: {total_orders}")
print(f"Average Order Value: ${avg_order_value:,.2f}")
print(f"Total Customers: {total_customers}")

# Revenue over time
revenue_over_time = df.groupby(df['order_date'].dt.to_period('M'))['total'].sum()
plt.figure(figsize=(12, 6))
revenue_over_time.plot()
plt.title('Monthly Revenue Trend')
plt.ylabel('Revenue')
plt.show()

# Data Visualization

Create visualizations like charts, graphs, and dashboards to represent insights from the analysis using libraries like matplotlib and seaborn.

In [ ]:
# Additional visualizations
# Top products by revenue
top_categories = df.groupby('category')['total'].sum().nlargest(10)
plt.figure(figsize=(10, 6))
top_categories.plot(kind='bar')
plt.title('Top 10 Categories by Revenue')
plt.xticks(rotation=45)
plt.show()

# Customer distribution by revenue
plt.figure(figsize=(10, 6))
sns.boxplot(x='cluster', y='total_revenue', data=customer_features)
plt.title('Revenue Distribution by Customer Segment')
plt.show()

# Rating distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='rating', data=df)
plt.title('Review Rating Distribution')
plt.show()

print("Analysis complete!")